# GUST-Flow demo
Run all cells in the `5dflow` environment. The workflow is: load data -> fixed 1000-step main A fit -> five-panel training GIF.
GT and segmentation are evaluation-only. Training uses the full-FOV PCMRASTD profile without segmentation cropping.
The right panel shows all Gaussians as faint 1-sigma projections and highlights their exact intersections with the current slice.
Positions, scales and rotations evolve during fitting. GIF diagnostics add overhead and are not a benchmark.


In [ ]:
from pathlib import Path
import sys
import h5py
import numpy as np
import torch
from IPython.display import Image, display

code = (Path.cwd() / "src").resolve()
if not code.is_dir():
    raise RuntimeError("Open this notebook from the GUST-Flow repository root")
sys.path.insert(0, str(code))
import gustflow
if not Path(gustflow.__file__).resolve().is_relative_to(code):
    raise RuntimeError("A different gustflow installation was loaded; restart the kernel")
from gustflow import GUSTFlow
from gustflow.visualization import SliceTrainingGif
assert torch.cuda.is_available(), "A CUDA GPU is required"


In [ ]:
data_path = Path("TestData.h5")
fold = 5
high_venc = np.array([150., 150., 150.], dtype=np.float32)
spacing = (2.5, 2.5, 2.5)  # FE, PE, SPE; update this for another dataset
num_iter = 1000
num_primitives = 8192

showv, showt, showz = 0, 3, None  # None selects the middle slice; indices start at zero
snapshot_every = 25
gif_path = Path("gustflow_training.gif")
ellipse_opacity = 0.18  # exact slice sections
show_all_projected = True
projection_opacity = 0.025  # faint full-depth context, not a slice intersection


In [ ]:
# Native H5 layout used by the PUDIP demo: Nv,Nt,SPE,PE,FE -> Nv,Nt,FE,PE,SPE. Read once.
with h5py.File(data_path, "r") as f:
    image = f["img"][:4].transpose(0, 1, 4, 3, 2)
    segmask = f["segmask"][:].transpose(2, 1, 0) if "segmask" in f else None

magnitude = np.abs(image).mean(axis=0)
phase_gt = np.angle(image[1:] * np.conj(image[:1])).astype(np.float32)
wrapped_phase = np.angle(np.exp(1j * phase_gt * fold)).astype(np.float32)
gt_velocity = phase_gt / np.pi * high_venc[:, None, None, None, None]
venc = high_venc / fold  # divide once
pcmra = magnitude * np.sqrt(np.sum(wrapped_phase**2, axis=0))
pcmra /= pcmra.max() + 1e-8
pcmra_std = pcmra.std(axis=0)  # center initialization uses std, not the temporal mean of weightmask
weightmask = (pcmra_std[None] * pcmra)[None].astype(np.float32)
weightmask /= weightmask.max() + 1e-8
showt = min(showt, wrapped_phase.shape[1] - 1)
showz = wrapped_phase.shape[-1] // 2 if showz is None else showz
assert 0 <= showv < 3 and 0 <= showt < wrapped_phase.shape[1] and 0 <= showz < wrapped_phase.shape[-1]
del image
print("Phase shape:", wrapped_phase.shape, "| reconstruction VENC:", venc)


In [ ]:
# No early stopping or best-iteration selection. Snapshots are for the GIF; the final output is always the last step.
video = SliceTrainingGif(channel=showv, frame=showt, slice_index=showz,
                         ellipse_opacity=ellipse_opacity, show_all_projected=show_all_projected,
                         projection_opacity=projection_opacity)
unwrapper = GUSTFlow(venc=venc, voxel_spacing=spacing,
                     num_iter=num_iter, num_primitives=num_primitives)
result = unwrapper.fit(
    wrapped_phase, weightmask, center_confidence=pcmra_std,
    gt_velocity=gt_velocity, segmask=segmask,
    eval_every=snapshot_every, callback=video,
)
recovered = result.recovered.detach().cpu().numpy()
print("Final iteration:", result.history["iterations_completed"])
print("Final NRMSE (evaluation mask):", result.history.get("returned_nrmse"))


In [ ]:
# Wrapped phase | GT | GUST-Flow | error | current GUST velocity + exact Gaussian sections
# Each ellipse is the exact intersection of a learned 3D 1-sigma Gaussian with the current slice.
# Faint full-depth projections show all primitives; brighter ellipses are exact slice sections.
video.save(result, wrapped_phase=wrapped_phase, gt_velocity=gt_velocity,
           venc=venc, segmask=segmask, spacing=spacing, path=gif_path, fps=8)
print(f"Saved {len(video.frames)} frames: {gif_path.resolve()}")
display(Image(filename=str(gif_path), width=1400))
